In [ ]:
from google.colab import drive
import json
import pandas as pd
import scipy.stats as st
import numpy as np
import io

drive.mount('/content/drive')
FILE_PATH_SU_DRIVE = "/content/drive/My Drive/Colab Notebooks/provaVec.json"

Connessione a Google Drive in corso...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Collegamento a Google Drive stabilito. Il file verrà letto da: /content/drive/My Drive/Colab Notebooks/provaVec.json


In [ ]:
#Estrazione, Trasformazione e Caricamento (ETL)

with open(FILE_PATH_SU_DRIVE, 'r') as f:
    data = json.load(f)

risultati_per_replica = []

PARAM_MAP = {
    '$0': 'm_SS1',
    '$1': 'm_SS2',
    '$2': 'm_arrivi',
    '$3': 'q_prob_U1',
    '$4': 'p_feedback'
}

PERCENTILI = {
    'Stima_Minimo_05perc': 0.05,
    'Stima_Massimo_95perc': 0.95
}

for run_id, run_data in data.items():
    attributes = run_data.get('attributes', {})
    config_name = attributes.get('experiment', 'N/A')

    try:
        repetition = int(attributes.get('repetition', -1))
    except ValueError:
        repetition = -1

    itervars_str = attributes.get('iterationvars', '')
    params = {}
    if itervars_str:
        try:
            pairs = itervars_str.split(', ')
            for pair in pairs:
                key, value = pair.split('=')
                col_name = PARAM_MAP.get(key, key)
                params[col_name] = value.replace('s', '')
        except Exception:
            pass

    row = {
        'Config': config_name,
        'repetition': repetition
    }
    row.update(params)

    vectors = run_data.get('vectors', [])

    if not vectors:
        continue

    for vec in vectors:
        metric_name = vec.get('name')

        if not (metric_name and metric_name.startswith('sojournTime')):
            continue

        raw_data = vec.get('value', [])

        # Prendiamo solo i valori (v), che sono negli indici dispari.
        if raw_data:
            values = [float(v) for v in raw_data[1::2]]
        else:
            values = []

        total_count = len(values)

        #Calcolo Metrica 3 (Percentili)
        #Implementa Eq. 6.115 (stat copia.pdf)
        if total_count == 0:
            for nome_stima in PERCENTILI.keys():
                row[f"{metric_name}_{nome_stima}"] = 0.0
        else:
            for nome_stima, p_val in PERCENTILI.items():
                percentile = np.quantile(values, p_val)
                row[f"{metric_name}_{nome_stima}"] = percentile

    risultati_per_replica.append(row)

df_risultati = pd.DataFrame(risultati_per_replica)
display(df_risultati.head())

Caricamento del file JSON da Google Drive (/content/drive/My Drive/Colab Notebooks/provaVec.json)...
ATTENZIONE: Questo richiederà diversi minuti, ma non devi caricarlo.
Trasformazione dei dati (ETL) per 9720 run...

Trasformazione completata.
Creato un DataFrame con 9720 righe (una per replica).


,Config,repetition,m_SS1,m_SS2,m_arrivi,q_prob_U1,p_feedback,sojournTimeU1_vec:vector_Stima_Minimo_05perc,sojournTimeU1_vec:vector_Stima_Massimo_95perc,sojournTimeU2_vec:vector_Stima_Minimo_05perc,sojournTimeU2_vec:vector_Stima_Massimo_95perc
0,Config1,7,2.4,4.0,4.0,0.4,0.8,11.024844,1045.827857,47508.145278,69432.952487
1,Config1,8,2.4,4.0,4.0,0.4,0.8,299.859185,5336.699883,0.000000,0.000000
2,Config1,9,2.4,4.0,4.0,0.4,0.8,23.980237,1135.504310,44975.934636,66761.799117
3,Config1,0,2.4,4.0,4.0,0.4,0.9,1512.285087,47724.322620,0.000000,0.000000
4,Config1,1,2.4,4.0,4.0,0.4,0.9,1359.479922,42748.133526,0.000000,0.000000


In [ ]:
#Calcolo Statistiche Finali (Stima Puntuale e CI)
#Con i valori per-replica (x_beta_m),
#calcolo della media e del CI su di essi (Eq. 6.117-6.119)
# -----------------------------------------------------------------

#Definizione delle colonne di configurazione
parametri_configurazione = ['Config', 'm_SS1', 'm_SS2', 'm_arrivi', 'q_prob_U1', 'p_feedback']

#Raggruppamento per le 486 configurazioni uniche
grouped = df_risultati.groupby(parametri_configurazione)

#Valore critico 't' per il 95% di confidenza con (n=20 -> df=19)
T_VALUE = st.t.ppf(0.975, 19)

#Calcolo delle statistiche aggregate
stima_puntuale = grouped.mean(numeric_only=True)
deviazione_standard = grouped.std(numeric_only=True)
n_campioni = grouped.count()

#Calcolo del margine di errore per l'Intervallo di Confidenza
margine_errore_ci = T_VALUE * (deviazione_standard / np.sqrt(n_campioni))

ci_limite_inferiore = stima_puntuale - margine_errore_ci
ci_limite_superiore = stima_puntuale + margine_errore_ci

#Rimozione della colonna 'repetition'
stima_puntuale = stima_puntuale.drop(columns='repetition')
ci_limite_inferiore = ci_limite_inferiore.drop(columns='repetition')
ci_limite_superiore = ci_limite_superiore.drop(columns='repetition')

#Stima Puntuale (Media dei 20 percentili)
display(stima_puntuale)

#nIntervallo di Confidenza (95%) - Limite Inferiore
display(ci_limite_inferiore)

#nIntervallo di Confidenza (95%) - Limite Superiore
display(ci_limite_superiore)

Inizio calcolo statistiche finali...

--- Analisi Metrica 3 Completata ---

Stima Puntuale (Media dei 20 percentili):


sojournTimeU1_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                             2.569714   
                                       0.8                                           109.450461   
                                       0.9                                          1575.561309   
                             0.6       0.6                                             3.065267   
                                       0.8                                          3848.303223   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          3872.044589   
                                       0.9                                          5780.483584   
                             0.8       0.6                                           183.054474   
                                       0.8                                          6903.141214   
                                       0.9                                          8557.861973   

                                                   sojournTimeU1_vec:vector_Stima_Massimo_95perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                  
Config1 2.4   2.0   4.0      0.4       0.6                                             51.612386   
                                       0.8                                           2632.955362   
                                       0.9                                          45108.804040   
                             0.6       0.6                                             83.496535   
                                       0.8                                          35077.693184   
...                                                                                          ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          35540.042863   
                                       0.9                                          51491.987110   
                             0.8       0.6                                           2064.895017   
                                       0.8                                          44666.663440   
                                       0.9                                          54216.688394   

                                                   sojournTimeU2_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                         10128.077383   
                                       0.8                                         25365.710637   
                                       0.9                                             0.000000   
                             0.6       0.6                                         14771.336719   
                                       0.8                                             0.000000   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                             0.000000   
                                       0.9                                             0.000000   
                             0.8       0.6                                         14298.800129   
                                       0.8                                             0.000000   
                                       0.9                                             0.000000   

                                                   sojournTimeU2_vec:vector_Stima_Massimo_95perc  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                          


Intervallo di Confidenza (95%) - Limite Inferiore:


sojournTimeU1_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                             2.550564   
                                       0.8                                            57.557379   
                                       0.9                                          1436.023544   
                             0.6       0.6                                             3.025819   
                                       0.8                                          3738.817171   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          3782.520781   
                                       0.9                                          5630.194077   
                             0.8       0.6                                           108.827490   
                                       0.8                                          6812.442010   
                                       0.9                                          8426.433200   

                                                   sojournTimeU1_vec:vector_Stima_Massimo_95perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                  
Config1 2.4   2.0   4.0      0.4       0.6                                             50.667831   
                                       0.8                                           1873.487430   
                                       0.9                                          44259.064114   
                             0.6       0.6                                             81.243574   
                                       0.8                                          34436.512694   
...                                                                                          ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          35080.509655   
                                       0.9                                          50904.932466   
                             0.8       0.6                                           1557.568012   
                                       0.8                                          44083.889843   
                                       0.9                                          53727.047671   

                                                   sojournTimeU2_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                          9970.866770   
                                       0.8                                         14809.636179   
                                       0.9                                             0.000000   
                             0.6       0.6                                         14184.203358   
                                       0.8                                             0.000000   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                             0.000000   
                                       0.9                                             0.000000   
                             0.8       0.6                                          4467.079484   
                                       0.8                                             0.000000   
                                       0.9                                             0.000000   

                                                   sojournTimeU2_vec:vector_Stima_Massimo_95perc  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                          


Intervallo di Confidenza (95%) - Limite Superiore:


sojournTimeU1_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                             2.588863   
                                       0.8                                           161.343542   
                                       0.9                                          1715.099074   
                             0.6       0.6                                             3.104716   
                                       0.8                                          3957.789275   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          3961.568396   
                                       0.9                                          5930.773092   
                             0.8       0.6                                           257.281457   
                                       0.8                                          6993.840418   
                                       0.9                                          8689.290746   

                                                   sojournTimeU1_vec:vector_Stima_Massimo_95perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                  
Config1 2.4   2.0   4.0      0.4       0.6                                             52.556941   
                                       0.8                                           3392.423294   
                                       0.9                                          45958.543967   
                             0.6       0.6                                             85.749496   
                                       0.8                                          35718.873675   
...                                                                                          ...   
Config2 3.5   4.0   6.0      0.6       0.8                                          35999.576072   
                                       0.9                                          52079.041754   
                             0.8       0.6                                           2572.222022   
                                       0.8                                          45249.437037   
                                       0.9                                          54706.329117   

                                                   sojournTimeU2_vec:vector_Stima_Minimo_05perc  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                                 
Config1 2.4   2.0   4.0      0.4       0.6                                         10285.287997   
                                       0.8                                         35921.785096   
                                       0.9                                             0.000000   
                             0.6       0.6                                         15358.470080   
                                       0.8                                             0.000000   
...                                                                                         ...   
Config2 3.5   4.0   6.0      0.6       0.8                                             0.000000   
                                       0.9                                             0.000000   
                             0.8       0.6                                         24130.520773   
                                       0.8                                             0.000000   
                                       0.9                                             0.000000   

                                                   sojournTimeU2_vec:vector_Stima_Massimo_95perc  
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                          

In [ ]:
stima_puntuale.to_csv("metrica3_stima_puntuale.csv")
ci_limite_inferiore.to_csv("metrica3_ci_inferiore.csv")
ci_limite_superiore.to_csv("metrica3_ci_superiore.csv")

files.download("metrica3_stima_puntuale.csv")
files.download("metrica3_ci_inferiore.csv")
files.download("metrica3_ci_superiore.csv")

Preparazione file per il download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download avviato.
